# 结构化数据：NumPy的结构化数组


虽然我们的数据通常可以很好地用同质值数组表示，但有时情况并非如此。本章演示了NumPy的*结构化数组*和*记录数组*的使用，这些数组为复合异构数据提供了高效存储。尽管这里展示的模式对简单操作很有用，但此类场景往往更适合使用Pandas的``DataFrame``，我们将在[第3部分](03.00-Introduction-to-Pandas.ipynb)中进行探讨。

In [1]:
import numpy as np

想象一下，我们有关于多个人的几类数据（例如，姓名、年龄和体重），我们希望将这些值存储以供Python程序使用。

可以将这些数据存储在三个单独的数组中：

In [2]:
name = ['Alice', 'Bob', 'Cathy', 'Doug']
age = [25, 45, 37, 19]
weight = [55.0, 85.5, 68.0, 61.5]

但这有些笨拙。这里没有任何信息表明这三个数组是相关的；NumPy 的结构化数组允许我们通过使用单一结构来更自然地存储所有这些数据。

回想一下，我们之前使用如下表达式创建了一个简单的数组：

In [3]:
x = np.zeros(4, dtype=int)

我们可以类似地使用复合数据类型规范创建结构化数组：

In [4]:
# Use a compound data type for structured arrays
data = np.zeros(4, dtype={'names':('name', 'age', 'weight'),
                          'formats':('U10', 'i4', 'f8')})
print(data.dtype)

[('name', '<U10'), ('age', '<i4'), ('weight', '<f8')]


这里 `'U10'` 翻译为“最大长度为 10 的 Unicode 字符串”，`'i4'` 翻译为“4 字节（即 32 位）整数”，而 `'f8'` 翻译为“8 字节（即 64 位）浮点数”。

我们将在接下来的部分讨论这些类型代码的其他选项。

现在，我们已经创建了一个空的容器数组，可以用我们的值列表填充该数组。

In [5]:
data['name'] = name
data['age'] = age
data['weight'] = weight
print(data)

[('Alice', 25, 55. ) ('Bob', 45, 85.5) ('Cathy', 37, 68. )
 ('Doug', 19, 61.5)]


如我们所希望的，数据现在方便地排列在一个结构化数组中。

结构化数组的便利之处在于，我们可以通过索引或名称来引用值。

In [6]:
# Get all names
data['name']

array(['Alice', 'Bob', 'Cathy', 'Doug'], dtype='<U10')

In [7]:
# Get first row of data
data[0]

np.void(('Alice', 25, 55.0), dtype=[('name', '<U10'), ('age', '<i4'), ('weight', '<f8')])

In [8]:
# Get the name from the last row
data[-1]['name']

np.str_('Doug')

使用布尔掩码，我们甚至可以进行一些更复杂的操作，例如按年龄过滤。



In [9]:
# Get names where age is under 30
data[data['age'] < 30]['name']

array(['Alice', 'Doug'], dtype='<U10')

如果您想进行比这些更复杂的操作，建议考虑使用Pandas包，该内容在[第4部分](04.00-Introduction-To-Matplotlib.ipynb)中进行了介绍。

正如您所看到的，Pandas提供了一个`DataFrame`对象，这是一个基于NumPy数组构建的结构，提供了多种有用的数据处理功能，与您在这里看到的类似，还有更多其他功能。

## 探索结构化数组的创建

结构化数组数据类型可以通过多种方式进行指定。

之前，我们看到了字典方法：

In [10]:
np.dtype({'names':('name', 'age', 'weight'),
          'formats':('U10', 'i4', 'f8')})

dtype([('name', '<U10'), ('age', '<i4'), ('weight', '<f8')])

为了清晰起见，数值类型可以使用 Python 类型或 NumPy `dtype` 来指定。

In [11]:
np.dtype({'names':('name', 'age', 'weight'),
          'formats':((np.str_, 10), int, np.float32)})

dtype([('name', '<U10'), ('age', '<i8'), ('weight', '<f4')])

复合类型也可以被指定为元组的列表：

In [12]:
np.dtype([('name', 'S10'), ('age', 'i4'), ('weight', 'f8')])

dtype([('name', 'S10'), ('age', '<i4'), ('weight', '<f8')])

如果类型的名称对您来说并不重要，您可以仅在以逗号分隔的字符串中指定类型。

In [13]:
np.dtype('S10,i4,f8')

dtype([('f0', 'S10'), ('f1', '<i4'), ('f2', '<f8')])

缩短的字符串格式代码可能并不直观，但它们是基于简单原则构建的。

第一个（可选）字符 `<` 或 `>` 分别表示“小端”或“大端”，并指定有效位的排序约定。

下一个字符指定数据类型：字符、字节、整数、浮点数等（见下表）。

最后一个或多个字符表示对象的大小，以字节为单位。

| Character    | Description           | Example                           |
| ---------    | -----------           | -------                           | 
| `'b'`        | Byte                  | `np.dtype('b')`                   |
| `'i'`        | Signed integer        | `np.dtype('i4') == np.int32`      |
| `'u'`        | Unsigned integer      | `np.dtype('u1') == np.uint8`      |
| `'f'`        | Floating point        | `np.dtype('f8') == np.int64`      |
| `'c'`        | Complex floating point| `np.dtype('c16') == np.complex128`|
| `'S'`, `'a'` | String                | `np.dtype('S5')`                  |
| `'U'`        | Unicode string        | `np.dtype('U') == np.str_`        |
| `'V'`        | Raw data (void)       | `np.dtype('V') == np.void`        |

## 更高级的复合类型

可以定义更高级的复合类型。

例如，您可以创建一个类型，其中每个元素包含一个值的数组或矩阵。

在这里，我们将创建一种数据类型，其 `mat` 组件由一个 $3\times 3$ 浮点矩阵组成。

In [14]:
tp = np.dtype([('id', 'i8'), ('mat', 'f8', (3, 3))])
X = np.zeros(1, dtype=tp)
print(X[0])
print(X['mat'][0])

(0, [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]])
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]


现在，`X` 数组中的每个元素由一个 `id` 和一个 $3\times 3$ 矩阵组成。

为什么要使用这个，而不是简单的多维数组或 Python 字典呢？

其中一个原因是，这种 NumPy `dtype` 可以直接映射到 C 结构定义，因此包含数组内容的缓冲区可以在适当编写的 C 程序中直接访问。

如果您正在为操作结构化数据的遗留 C 或 Fortran 库编写 Python 接口，结构化数组可以提供强大的接口。

## 记录数组：结构化数组的新特性

NumPy 还提供了记录数组（`np.recarray` 类的实例），这些数组与刚才描述的结构化数组几乎相同，但具有一个额外的特点：字段可以作为属性访问，而不是作为字典键。

请回忆一下，我们之前通过以下方式访问样本数据集中的年龄：

In [15]:
data['age']

array([25, 45, 37, 19], dtype=int32)

如果我们将数据视为记录数组，则可以用稍微少一些的按键访问它：

In [16]:
data_rec = data.view(np.recarray)
data_rec.age

array([25, 45, 37, 19], dtype=int32)

缺点是，对于记录数组，在访问字段时会有一些额外的开销，即使使用相同的语法。

In [17]:
%timeit data['age']
%timeit data_rec['age']
%timeit data_rec.age

33.3 ns ± 0.364 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)
336 ns ± 2.43 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
667 ns ± 5.59 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


更方便的符号是否值得（略微的）开销将取决于您自己的应用。

## 进入Pandas

本章关于结构化数组和记录数组的内容特意放在本书这一部分的末尾，因为它为我们将要讨论的下一个包：Pandas，做好了很好的铺垫。

在某些情况下，结构化数组非常有用，例如当您使用NumPy数组映射到C、Fortran或其他语言中的二进制数据格式时。

然而，对于日常使用结构化数据而言，Pandas包是一个更好的选择；我们将在接下来的章节中深入探讨它。